# TP 1 — Environnement, Git et les limites du traitement local

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

Ce notebook guide les parties **C** (génération des données), **D** (exploration
Pandas) et **E** (montée en charge) du TP 1. Complétez les cellules marquées
`À COMPLÉTER`, exécutez tout de bout en bout, puis poussez le notebook **avec
ses sorties** dans `notebooks/` de votre dépôt Git.

> Réflexe d'ingénieur : on ne dit pas « c'est lent », on dit « 12,4 s pour
> 50 000 lignes ». **Toutes** vos observations doivent être chiffrées.

## 0. Vérification de l'environnement

La cellule suivante doit s'exécuter sans erreur et afficher des versions
cohérentes (Python ≥ 3.9). Sinon, retournez à la partie A du TP
(`python3 check_env.py`).

In [1]:
import sys, platform
import pandas as pd
import numpy as np

print("Python :", sys.version.split()[0], "—", platform.system())
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

Python : 3.12.13 — Linux
pandas : 2.2.2
numpy  : 2.0.2


### Tableau de relevés

Vous remplirez ce dictionnaire au fil des exercices ; il sert de source unique
pour la partie E et pour votre `DIAGNOSTIC.md`.

In [6]:
DATA_DIR = "./data"

releves = {
    # fichier            : {"lignes": ..., "temps_s": ..., "memoire_Mo": ..., "disque_Mo": ...}
    "customers.csv"      : {},
    "orders.csv"         : {},
    "orders+items (jointure)" : {},
    "events.json"        : {},
}

## Exercice 1 — `customers.csv` : premier contact

Chargez le fichier, puis relevez :
1. ses **dimensions** (lignes × colonnes) ;
2. le **type** de chaque colonne (`dtypes`) ;
3. un **aperçu** (`head`) ;
4. son **empreinte mémoire** en Mo — utilisez
   `df.memory_usage(deep=True).sum()` (pourquoi `deep=True` ? voyez la
   question 1.b).

In [7]:
# === À COMPLÉTER ===
import os, time

t0 = time.perf_counter()
customers = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"))                     # charger DATA_DIR/customers.csv
t1 = time.perf_counter()

print("Dimensions :", customers.shape)
print("Temps de chargement : %.2f s" % (t1 - t0))
print(customers.dtypes)
customers.head()

Dimensions : (50250, 10)
Temps de chargement : 0.23 s
customer_id         object
prenom              object
nom                 object
email               object
telephone           object
adresse             object
ville               object
region              object
date_naissance      object
date_inscription    object
dtype: object


,customer_id,prenom,nom,email,telephone,adresse,ville,region,date_naissance,date_inscription
0,C001641,Omar,Mbaye,omar.mbaye71@ucad.edu.sn,770243847,avenue Masse,Thiès,Thiès,2004-03-07,2022-08-25
1,C025699,Yacine,Kane,yacine.kane38@yahoo.fr,+221777875632,"14, chemin Duhamel",THIÈS,Thiès,1974-07-28,2022-01-14
2,C032225,Fatoumata,Wade,NaN,+221 75 201 72 55,"96, rue Lucie Petit",Thiès,Thiès,1975-12-06,2025-06-08
3,C032075,Ndèye,Thiam,ndeye.thiam908@ucad.edu.sn,+221786471546,avenue Guillaume Couturier,Dakar,Dakar,1976-11-19,2026-05-05
4,C048566,Seynabou,Sow,seynabou.sow939@orange.sn,+221 77 513 98 37,avenue de Morvan,Touba,Diourbel,2005-07-29,2024-02-25


**Question 1.a** — Remplissez la ligne `customers.csv` du tableau de relevés.

In [8]:
# === À COMPLÉTER ===
mem_Mo = customers.memory_usage(deep=True).sum() / 1024**2                       # mémoire du DataFrame en Mo (deep=True)
disque_Mo = os.path.getsize(os.path.join(DATA_DIR, "customers.csv")) / 1024**2    # taille du fichier sur disque en Mo
releves["customers.csv"] = {"lignes": len(customers), "temps_s": round(t1 - t0, 2),
                            "memoire_Mo": round(mem_Mo, 1), "disque_Mo": round(disque_Mo, 1)}
releves["customers.csv"]

{'lignes': 50250,
 'temps_s': 0.23,
 'memoire_Mo': np.float64(29.6),
 'disque_Mo': 5.7}

**Question 1.b** *(markdown — répondez ici)* — Comparez
`memory_usage()` **sans** puis **avec** `deep=True` sur `customers`.
Pourquoi un tel écart ? Quel est le rapport mémoire/disque ? Notez votre
explication : elle resservira mot pour mot dans le diagnostic.

*Votre réponse :* Sans `deep=True`, `memory_usage()` ne comptabilise pas complètement la mémoire occupée par les chaînes de texte et les objets Python. Avec `deep=True`, elle prend en compte la taille réelle de ces valeurs, ce qui explique l’écart important. La mémoire utilisée par le DataFrame est donc bien supérieure à la taille du fichier sur disque, car le CSV est stocké en texte compact alors que Pandas charge les données dans des structures Python plus lourdes. Le rapport mémoire/disque est donc supérieur à 1, environ 5 fois plus de mémoire que le fichier sur disque.

## Exercice 2 — `orders.csv` : statistiques et première surprise

Chargez `orders.csv` en **chronométrant**, puis :
1. nombre de commandes par `statut` ;
2. nombre de commandes par `canal` ;
3. essayez de calculer le **montant total** des commandes
   (`montant_total_fcfa`)… que se passe-t-il ? Regardez le `dtype` de la
   colonne. **Ne corrigez pas** : documentez (section 6).

In [9]:
# === À COMPLÉTER ===
import os, time

t0 = time.perf_counter()
orders = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))  # charger DATA_DIR/orders.csv
t1 = time.perf_counter()
print("Temps : %.2f s — %d lignes" % (t1 - t0, len(orders)))

print("Commandes par statut :")
print(orders["statut"].value_counts())

print("\nCommandes par canal :")
print(orders["canal"].value_counts())

print("\nType de montant_total_fcfa :", orders["montant_total_fcfa"].dtype)
try:
    print("Somme :", orders["montant_total_fcfa"].sum())
except Exception as e:
    print(type(e).__name__, e)

Temps : 1.29 s — 500000 lignes
Commandes par statut :
statut
livrée       389865
annulée       44842
en_cours      40237
retournée     25056
Name: count, dtype: int64

Commandes par canal :
canal
mobile_app    324565
web           175435
Name: count, dtype: int64

Type de montant_total_fcfa : object
Somme : 2027001000028000322000450005750025900040500282000900084000195004399002875002065003900014280075004400087550068000465001250006400075150039100625500144500750023600056000103000930001045002585001600014160057800910001840006120002600051000745005500350048500933007300671600506003000008900033200045008800039000230007150029100204900014650024650011750060002457006460001800020800650044000255008533001100025660055002045003000245004050023000215001475006775006190036000550011500650002300059800122700515001564004050024550045001760015150047600320200780008500040004800026170009500028500125000735008000066000011630087000134800228500550095001545006833003600055003465005220075350013600585000690001880001295005450

**Question 2.a** — Remplissez la ligne `orders.csv` du tableau de relevés
(lignes, temps, mémoire deep, taille disque).

In [10]:
# === À COMPLÉTER ===
mem_Mo = orders.memory_usage(deep=True).sum() / 1024**2
disque_Mo = os.path.getsize(os.path.join(DATA_DIR, "orders.csv")) / 1024**2

releves["orders.csv"] = {"lignes": len(orders), "temps_s": round(t1 - t0, 2),
                            "memoire_Mo": round(mem_Mo, 1), "disque_Mo": round(disque_Mo, 1)}
releves["orders.csv"]

{'lignes': 500000,
 'temps_s': 1.29,
 'memoire_Mo': np.float64(176.4),
 'disque_Mo': 31.0}

## Exercice 3 — Jointure `orders` × `order_items`

Les jointures sont au cœur de l'analytique — et elles **coûtent cher**.
1. Chargez `order_items.csv`.
2. Mesurez la mémoire totale **avant** la jointure (somme des deux DataFrames).
3. Réalisez la jointure (`merge` sur `order_id`), chronométrez-la.
4. Mesurez la mémoire du résultat. Conclusion ?

In [11]:
# === À COMPLÉTER ===
items = pd.read_csv(os.path.join(DATA_DIR, "order_items.csv"))  # charger order_items.csv
mem_avant = orders.memory_usage(deep=True).sum() / 1024**2 + items.memory_usage(deep=True).sum() / 1024**2  # mémoire orders + items (Mo, deep)

t0 = time.perf_counter()
joint = orders.merge(items, on="order_id")  # merge orders x items sur order_id
t1 = time.perf_counter()

mem_joint = joint.memory_usage(deep=True).sum() / 1024**2  # mémoire du résultat (Mo, deep)
orders_disk = os.path.getsize(os.path.join(DATA_DIR, "orders.csv")) / 1024**2
items_disk = os.path.getsize(os.path.join(DATA_DIR, "order_items.csv")) / 1024**2
releves["orders+items (jointure)"] = {"lignes": len(joint), "temps_s": round(t1 - t0, 2),
                            "memoire_Mo": round(mem_joint, 1), "disque_Mo": round(orders_disk + items_disk, 1)}
releves["orders+items (jointure)"]

print("Jointure : %.2f s — %d lignes" % (t1 - t0, len(joint)))
print("Mémoire avant : %.0f Mo — résultat seul : %.0f Mo" % (mem_avant, mem_joint))

Jointure : 0.88 s — 1129775 lignes
Mémoire avant : 377 Mo — résultat seul : 538 Mo


## Exercice 4 — `events.json` : le gros morceau

`events.json` est au format **JSON Lines** (un objet JSON par ligne).
À l'échelle 0.1 il reste chargeable (~330 000 lignes) ; mesurez précisément :
temps, mémoire deep, taille disque, et le **ratio mémoire/disque**.

In [12]:
# === À COMPLÉTER ===
t0 = time.perf_counter()
events = pd.read_json(os.path.join(DATA_DIR, "events.json"), lines=True)
t1 = time.perf_counter()

mem_events = events.memory_usage(deep=True).sum() / 1024**2
disque_events = os.path.getsize(os.path.join(DATA_DIR, "events.json")) / 1024**2

releves["events.json"] = {"lignes": len(events), "temps_s": round(t1 - t0, 2),
                            "memoire_Mo": round(mem_events, 1), "disque_Mo": round(disque_events, 1)}
releves["events.json"]

{'lignes': 3301501,
 'temps_s': 28.5,
 'memoire_Mo': np.float64(1346.5),
 'disque_Mo': 653.3}

## Exercice 5 — Synthèse des relevés (échelle 1.0)

In [13]:
pd.DataFrame(releves).T

,lignes,temps_s,memoire_Mo,disque_Mo
customers.csv,50250.0,0.23,29.6,5.7
orders.csv,500000.0,1.29,176.4,31.0
orders+items (jointure),1129775.0,0.88,537.9,67.3
events.json,3301501.0,28.50,1346.5,653.3


## Partie E — Montée en charge : échelle 1.0 sur Colab

**À faire sur Google Colab** (pas sur votre laptop) :

1. Téléversez `generate_data.py` et `requirements.txt` dans la session ;
2. `!pip install -r requirements.txt` puis
   `!python3 generate_data.py --scale 1.0 --outdir ./data` (~2 min, 860 Mo) ;
3. Ré-exécutez les exercices 2 à 4 sur ces données (adaptez `DATA_DIR`) ;
4. Tentez le chargement intégral de `events.json` (≈ 685 Mo, 3,3 M lignes)
   et **observez** : durée, occupation RAM (panneau « RAM » de Colab),
   avertissements, voire crash du noyau.

> Un crash du noyau **est un résultat de mesure**, pas un échec : notez
> précisément ce qui s'est passé et à quel moment.

In [14]:
# Cellule Colab (échelle 1.0) — recopiez vos mesures ci-dessous en local
releves_colab = {
    "orders.csv (1.0)"   : {"lignes": 500000.0, "temps_s": 1.23, "memoire_Mo": 176.4},
    "events.json (1.0)"  : {"lignes": 3301501.0, "temps_s": 35.10, "memoire_Mo": 1346.5,
                            "issue": "chargé / averti / crash ?"},
}
releves_colab

{'orders.csv (1.0)': {'lignes': 500000.0,
  'temps_s': 1.23,
  'memoire_Mo': 176.4},
 'events.json (1.0)': {'lignes': 3301501.0,
  'temps_s': 35.1,
  'memoire_Mo': 1346.5,
  'issue': 'chargé / averti / crash ?'}}

1. Téléversement generate_data.py et requirements.txt dans la session

In [2]:
from google.colab import files

print("chargement du fichier generate_data.py")
uploaded_data = files.upload()

chargement du fichier generate_data.py


Saving generate_data.py to generate_data.py


In [3]:
print("chargement du fichier 'requirements.txt':")
uploaded_reqs = files.upload()

chargement du fichier 'requirements.txt':


Saving requirements.txt to requirements.txt


2. !pip install -r requirements.txt puis !python3 generate_data.py --scale 1.0 --outdir ./data (~2 min, 860 Mo)

In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 80.3 MB/s eta 0:00:00
  Attempting uninstall: jupyter-server
    Found existing installation: jupyter_server 2.18.2
    Uninstalling jupyter_server-2.18.2:
      Successfully uninstalled jupyter_server-2.18.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires jupyter-server==2.18.2, but you have jupyter-server 2.20.0 

In [5]:
!python3 generate_data.py --scale 1.0 --outdir ./data

Génération (graine=42, échelle=1.0) vers ./data/
  - customers : 50,000 lignes
  - products  : 2,000 lignes
  - orders    : 500,000 lignes (+ lignes de commandes)
  - payments  : ~441,514 lignes (JSON Lines)
  - events    : ~3,300,000 lignes (JSON Lines, écrit par lots)
  - pg_init.sql : référentiel propre (customers, products)

Terminé. Fichiers générés :
  anomalies_manifest.json             0.0 Mo
  customers.csv                       6.0 Mo
  events.json                       685.0 Mo
  order_items.csv                    38.1 Mo
  orders.csv                         32.5 Mo
  payments.json                      90.3 Mo
  pg_init.sql                         6.1 Mo
  products.csv                        0.1 Mo


### Extrapolation

Complétez à partir de vos deux points de mesure (0.1 et 1.0), en supposant
d'abord une croissance **linéaire** :

| Scénario | `events.json` | Temps estimé | Mémoire estimée | Tenable sur 1 machine ? |
|---|---|---|---|---|
| TP échelle 0.1 (mesuré) | ≈ 70 Mo |2.91 s | 137.7 MO | Oui |
| TP échelle 1.0 (mesuré) | ≈ 685 Mo | 35.10 s | 1346.5 MO | Non, risque de crash mémoire |
| Plateforme réelle, 1 an | ≈ 50 Go | 2624 s (~44 min) | ≈ 98 Go | Non |
| Grand acteur régional | ≈ 1 To | 53738 s (~15 h) | ≈ 2.0 To | Non |

**Questions** (réponses dans `DIAGNOSTIC.md`) :
1. L'extrapolation linéaire est-elle optimiste ou pessimiste ? Pourquoi ?
2. Quelles parades *sans* distribution connaissez-vous (chunks, `dtype`
   optimisés, échantillonnage, formats binaires) — et jusqu'où
   repoussent-elles le mur ?
3. À partir de quel point le **scale-out** devient-il inévitable?

## 6. Anomalies repérées (sans les corriger !)

Listez ici, **avec une preuve chiffrée** (un comptage, un exemple), les
bizarreries rencontrées. Elles seront traitées en séances 3 et 9.

In [ ]:
# === À COMPLÉTER ===
# Exemples de pistes (à vérifier et chiffrer) :
# - emails vides ou "N/A" dans customers
# - villes en casse/accents incohérents
# - dtype de orders["montant_total_fcfa"]
# - dates de naissance aberrantes
...

### 6.1. Anomalies dans la colonne `email` de `customers`

In [15]:
import numpy as np

# Emails vides (NaN)
empty_emails = customers['email'].isnull().sum()
print(f"Nombre d'emails vides (NaN): {empty_emails}")

# Emails contenant 'N/A' (insensible à la casse)
na_emails = customers['email'].astype(str).str.contains('N/A', case=False, na=False).sum()
print(f"Nombre d'emails contenant 'N/A': {na_emails}")

print(f"Total d'anomalies d'email: {empty_emails + na_emails}")

Nombre d'emails vides (NaN): 1506
Nombre d'emails contenant 'N/A': 0
Total d'anomalies d'email: 1506


### 6.2. Incohérences de casse/accents dans les noms de villes

In [16]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.2 MB/s eta 0:00:00


In [17]:
from unidecode import unidecode

# Fonction pour normaliser les noms de ville (minuscule, sans accents)
def normalize_city(city):
    if isinstance(city, str):
        return unidecode(city).lower().strip()
    return city

# Appliquer la normalisation et comparer avec les originaux
customers['city_normalized'] = customers['ville'].apply(normalize_city)

inconsistent_cities = customers.groupby('city_normalized')['ville'].nunique() > 1
inconsistent_city_names = inconsistent_cities[inconsistent_cities].index.tolist()

print(f"Nombre de villes avec des incohérences de casse/accents: {len(inconsistent_city_names)}")
print("Exemples de villes incohérentes (normalisé -> valeurs originales):")
for city_norm in inconsistent_city_names[:5]: # Afficher les 5 premiers exemples
    original_names = customers[customers['city_normalized'] == city_norm]['ville'].unique()
    print(f"- {city_norm}: {list(original_names)}")

# Nettoyer la colonne temporaire
del customers['city_normalized']

Nombre de villes avec des incohérences de casse/accents: 19
Exemples de villes incohérentes (normalisé -> valeurs originales):
- dakar: ['Dakar', 'dakar', 'DAKAR']
- diourbel: ['Diourbel', 'diourbel', 'DIOURBEL']
- fatick: ['Fatick', 'FATICK', 'fatick']
- guediawaye: ['Guédiawaye', 'Guediawaye', 'GUÉDIAWAYE', 'guédiawaye', 'guediawaye']
- kaffrine: ['Kaffrine', 'KAFFRINE', 'kaffrine']


### 6.3. `dtype` de `orders["montant_total_fcfa"]`

In [18]:
print(f"Le dtype de la colonne 'montant_total_fcfa' est: {orders['montant_total_fcfa'].dtype}")

# Tenter de convertir en numérique pour voir les erreurs
# Note: Ceci est pour la détection d'anomalies, pas pour une correction permanente ici.
non_numeric_values = pd.to_numeric(orders['montant_total_fcfa'], errors='coerce').isnull().sum()

print(f"Nombre de valeurs non numériques dans 'montant_total_fcfa': {non_numeric_values}")
if non_numeric_values > 0:
    print("Ceci indique que la colonne contient des caractères non numériques empêchant une conversion directe en type numérique.")
    # Afficher quelques exemples de valeurs problématiques
    problematic_examples = orders[pd.to_numeric(orders['montant_total_fcfa'], errors='coerce').isnull()]['montant_total_fcfa'].unique()
    print(f"Exemples de valeurs problématiques: {problematic_examples[:5]}")

Le dtype de la colonne 'montant_total_fcfa' est: object
Nombre de valeurs non numériques dans 'montant_total_fcfa': 5000
Ceci indique que la colonne contient des caractères non numériques empêchant une conversion directe en type numérique.
Exemples de valeurs problématiques: ['7000 FCFA' '62500 FCFA' '201000 FCFA' '453500 FCFA' '45500 FCFA']


### 6.4. Dates de naissance aberrantes

In [19]:
import pandas as pd
from datetime import datetime

# Convertir en format datetime, avec gestion des erreurs
customers['date_naissance_dt'] = pd.to_datetime(customers['date_naissance'], errors='coerce')

# Compter les dates de naissance invalides (NaT après conversion)
invalid_birth_dates = customers['date_naissance_dt'].isnull().sum()
print(f"Nombre de dates de naissance invalides: {invalid_birth_dates}")

# Identifier les dates de naissance dans le futur
future_birth_dates = customers[customers['date_naissance_dt'] > datetime.now()]['date_naissance_dt'].count()
print(f"Nombre de dates de naissance dans le futur: {future_birth_dates}")

# Identifier les dates de naissance très anciennes (ex: avant 1900)
ancient_birth_dates = customers[customers['date_naissance_dt'] < pd.Timestamp('1900-01-01')]['date_naissance_dt'].count()
print(f"Nombre de dates de naissance antérieures à 1900: {ancient_birth_dates}")

# Nettoyer la colonne temporaire
del customers['date_naissance_dt']

Nombre de dates de naissance invalides: 0
Nombre de dates de naissance dans le futur: 126
Nombre de dates de naissance antérieures à 1900: 0


## 7. Vers le diagnostic

Avant de fermer ce notebook, vérifiez que vous savez répondre, **mesures à
l'appui**, aux trois questions du canevas `docs/DIAGNOSTIC.md` :

1. **Constats** — où, précisément, le traitement local a-t-il atteint ses
   limites (chiffres) ?
2. **Analyse** — pourquoi (mémoire vs disque, jointures, croissance des
   volumes) ?
3. **Besoins** — qu'attend-on d'une architecture distribuée (et que
   garde-t-on de Pandas) ?

Puis : `git add notebooks/ docs/ && git commit -m "TP1 : exploration et
diagnostic" && git push`.